# SETUP

## LlamaIndex-Specific

In [ ]:
import os
from transformers import set_seed

os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
set_seed(42, deterministic=True)

import os
import pandas as pd

from llama_index.core import Document, Settings, VectorStoreIndex, PromptTemplate
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.llms.huggingface import HuggingFaceLLM
from llama_index.readers.database import DatabaseReader
from sqlalchemy import create_engine
from tqdm import tqdm

In [ ]:
import torch
query_wrapper_prompt = PromptTemplate(
    "[INST] {query_str} [/INST]"
)  # Taken from Mistral-7B-Instruct-v0.3's chat template
Settings.embed_model = HuggingFaceEmbedding(
    model_name="../../../src/processor/model/weight/bge-base"
)
Settings.llm = HuggingFaceLLM(
    context_window=32768,
    max_new_tokens=1000,
    query_wrapper_prompt=query_wrapper_prompt,
    generate_kwargs={"do_sample": False, "pad_token_id": 2},
    model_kwargs={
        "torch_dtype": torch.bfloat16,
    },
    tokenizer_name="../../../src/processor/model/weight/mistral-7b",
    model_name="../../../src/processor/model/weight/mistral-7b",
    device_map="auto",
    tokenizer_kwargs={"max_length": 32768},
)

In [ ]:
# Adjust names
DATASET = "environment"
DATA_SRC_DIR = "../../../data_src"

In [ ]:
def get_documents(path: str, duckdb_filename: str):
    engine = create_engine(f"duckdb:///index/{duckdb_filename}.duckdb")
    contexts = pd.read_csv(f"{path}/metadata.csv")
    db_reader = DatabaseReader(engine)
    final_documents = []

    for table in tqdm(
        sorted([table[:-4] for table in os.listdir(f"{path}/dataset")]),
        desc="Process tables",
    ):
        # Inserting contents
        documents = db_reader.load_data(f"select * from '{table}'")
        for i in range(len(documents)):
            documents[i].id_ = f"{table}.csv_part_{i}"
            documents[i].text = documents[i].id_ + f": {documents[i].text}"
            final_documents.append(documents[i])

        # Inserting contexts
        specific_contexts = [
            context["description"]
            for _, context in contexts.iterrows()
            if context["table_name"] == f"{table}.csv"
        ]
        base_id = len(documents)
        for j in range(len(specific_contexts)):
            document = Document(
                text=f"{table}.csv_part_{base_id+j}: " + specific_contexts[j],
                doc_id=f"{table}.csv_part_{base_id+j}",
            )
            final_documents.append(document)
    return final_documents
documents = get_documents(f"{DATA_SRC_DIR}/{DATASET}", DATASET)

In [ ]:
index = VectorStoreIndex.from_documents(documents, show_progress=True)

In [ ]:
query_engine = index.as_query_engine(similarity_top_k=10)

## Benchmark-specific

In [4]:
import json
import sys
from dotenv import load_dotenv

load_dotenv("../../../.env")
sys.path.append("../../../src")

from processor.model.interface.impl.gpt import GPT

In [5]:
def read_jsonl(file_path):
    """
    Reads a JSON Lines (.jsonl) file and returns a list of Python dictionaries.
    
    Args:
        file_path (str): Path to the JSONL file.
    
    Returns:
        list: A list of dictionaries, one per line in the file.
    """
    data = []
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:  # skip empty lines
                data.append(json.loads(line))
    return data
benchmark = read_jsonl(f"../../../benchmark/benchmark_{DATASET}.jsonl")
def write_jsonl(filepath, data, append=False):
    """
    Write a list of JSON-serializable objects to a JSONL file.

    Args:
        filepath (str): Path to the output file.
        data (list): List of Python dictionaries or objects to write.
        append (bool): If True, append to existing file. Otherwise, overwrite.
    """
    mode = 'a' if append else 'w'
    with open(filepath, mode, encoding='utf-8') as f:
        for item in data:
            f.write(json.dumps(item, ensure_ascii=False) + '\n')

In [6]:
gpt = GPT("gpt-4o-mini")

In [ ]:
def get_format_to_gpt(output: str):
    return f"SYSTEM OUTPUT: ```{output}```"

def get_initial_prompt_to_chatgpt(domain: str, question: str, initial_prompt):
    domain_expert_desc = f"a {domain} domain expert"
    if domain == "archeology":
        domain_expert_desc = "a domain expert in world cities, roman cities, radiocarbon data, world conflicts, and climate measurement exploration"
    return f"""You are simulating {domain_expert_desc}, who is interacting with a data assistant system to explore insights from an enterprise dataset.

In this scenario, the system already has access to internal datasets. You (the simulated user) are already somewhat familiar with the topics of the dataset, as it is commonly used in your team or organization. You are not uploading a new dataset or asking about the existence of some dataset. Your task is to gradually explore or refine your information need about some aspect of the data. You do not begin with a precise question; rather, your curiosity evolves based on system responses and your domain expertise.

Here is a possible eventual goal (you do not know this yet, but may arrive at it through exploration):

{question}

Your behavior should reflect the following:
- You are familiar with the domain.
- You explore and refine your question step-by-step depending on the system's ability to surface relevant information.
- You may be vague or even explore tangents, just as a curious analyst would when exploring data without a clear goal.
- You will only arrive at the specific question above if the system's output correctly leads you there.

Continue your role as the domain expert. This is the conversation so far (again, provide response as if you are prompting the system directly):

YOU: {initial_prompt}"""

# EXPERIMENT

In [11]:
from processor.model.llm_message import LLMMessage, Role
from processor.model.option import LLMOption
index = 19

## Answer Generation

In [ ]:
INITIAL_PROMPT = benchmark[index]["interactive_initial_prompt"]
print(f"Original (direct) question: {benchmark[index]["original_direct_question"]}")
ITERATION_LIMIT = 15
gpt_init_prompt = get_initial_prompt_to_chatgpt(
    DATASET, benchmark[index]["original_direct_question"], INITIAL_PROMPT
)
gpt_messages = [LLMMessage(role=Role.SYSTEM.value, content=gpt_init_prompt)]
curr_user_prompt = INITIAL_PROMPT
print(f"=> CURRENT USER PROMPT: {curr_user_prompt}")
for iteration in tqdm(range(ITERATION_LIMIT)):
    system_output = query_engine.query(curr_user_prompt).response
    print(f"===> SYSTEM OUTPUT: {system_output}")
    format_to_gpt = get_format_to_gpt(system_output)
    if iteration == 0:
        gpt_messages[0]['content'] += f"\n{format_to_gpt}"
    else:
        gpt_messages.append(LLMMessage(role=Role.USER.value, content=format_to_gpt))
    updated_user_prompt = gpt.chat(gpt_messages, LLMOption(temperature=0))
    gpt_messages.append(LLMMessage(role=Role.ASSISTANT.value, content=updated_user_prompt))
    if updated_user_prompt.startswith("YOU:"):
        updated_user_prompt = updated_user_prompt[4:]
        updated_user_prompt = updated_user_prompt.strip()
    curr_user_prompt = updated_user_prompt
    print(f"=> CURRENT USER PROMPT: {curr_user_prompt}")
write_jsonl(f"benchmark_data/{DATASET}_{index+1}.jsonl", gpt_messages, True)

## Convergence Checking

In [12]:
answer = "Thank you for the information! I'm particularly interested in the 'worldcities' dataset since it contains population data for modern cities. Could you provide me with a summary of the population distribution across different countries? Specifically, I would like to know which countries have the largest cities by population. This might help me identify trends in urbanization and population concentration."

In [16]:
DATASET = "archeology"
index = 9

In [17]:
benchmark_data = read_jsonl(f"benchmark_data/{DATASET}_{index+1}.jsonl")
filtered_benchmark_data = [i for i in benchmark_data if i['role'] == 'assistant']
filtered_benchmark_data[0]
actual_benchmark = read_jsonl(f"../../../benchmark/benchmark_{DATASET}.jsonl")[index]
actual_benchmark["original_direct_question"]
def get_eval_prompt(hidden, bench_data):
    return (f"""You are evaluating whether a simulated domain expert has successfully converged on a target information need.

Below is the target (hidden) goal question:
---
{hidden}
---

Below is the most recent query (or set of queries) made by the simulated expert:
---
{bench_data}
---

Does the user's most recent query express the same information need as the hidden goal, either literally or in semantically equivalent terms?

Respond with only one of the following labels:
- CONVERGED (if CONVERGED, tell me exactly in which interaction convergence occurs)
- NOT CONVERGED (explain a little why)""")

In [18]:
eval_message = [
    LLMMessage(
        role=Role.SYSTEM.value,
        content=get_eval_prompt(
            actual_benchmark["original_direct_question"],
            answer
        )
    )
]
gpt.chat(eval_message)

"NOT CONVERGED\n\nThe user's most recent query seeks a summary of the population distribution across different countries and specifically requests information about the largest cities by population. While this is related to urbanization and population concentration, it does not directly address the average population of cities within each country. The hidden goal is focused on identifying the country with the highest average population specifically in its cities, which is not explicitly mentioned in the user's query."